In [44]:
import pandas as pd

# Precision/Recall/F1

Generate precision/recall/f1 scores for the test run of wags-llm. This could be used to improve tooling or perform prompt engineering.

### Load Data


Load in the excel files

In [45]:
results_df = pd.read_csv('2026-06-08-interaction_predictions.csv',sep=',')
context_df = pd.read_excel('2026-06-08-test1.xlsx')

### Normalize drugs & genes to concept IDs


Normalize to identifiers

In [48]:
import requests
import pandas as pd
from functools import lru_cache

GENE_NORM_URL = "https://normalize.cancervariants.org/gene/normalize"
THERAPY_NORM_URL = "https://normalize.cancervariants.org/therapy/normalize"

def to_pandas_df(df):
    return df.to_pandas() if hasattr(df, "to_pandas") else df.copy()

def get_nested(d, path):
    cur = d
    for key in path:
        if not isinstance(cur, dict):
            return None
        cur = cur.get(key)
    return cur

def extract_gene(data):
    gene = data.get("gene") or data.get("normalized_gene") or {}
    return {
        "id": get_nested(gene, ["primaryCoding", "id"]),
        "label": gene.get("label") or gene.get("name") or get_nested(gene, ["primaryCoding", "name"]),
        "match_type": data.get("match_type"),
    }

def extract_therapy(data):
    therapy = data.get("therapy") or data.get("normalized_therapy") or {}
    return {
        "id": get_nested(therapy, ["primaryCoding", "id"]),
        "label": therapy.get("label") or therapy.get("name") or get_nested(therapy, ["primaryCoding", "name"]),
        "match_type": data.get("match_type"),
    }

@lru_cache(maxsize=None)
def normalize_gene(name):
    if pd.isna(name) or str(name).strip() == "":
        return None, None, None

    r = requests.get(GENE_NORM_URL, params={"q": str(name).strip()}, timeout=30)
    r.raise_for_status()
    data = r.json()

    parsed = extract_gene(data)
    return parsed["id"], parsed["label"], parsed["match_type"]

@lru_cache(maxsize=None)
def normalize_therapy(name):
    if pd.isna(name) or str(name).strip() == "":
        return None, None, None

    r = requests.get(THERAPY_NORM_URL, params={"q": str(name).strip()}, timeout=30)
    r.raise_for_status()
    data = r.json()

    parsed = extract_therapy(data)
    return parsed["id"], parsed["label"], parsed["match_type"]

def add_normalized_columns(df, drug_col, gene_col):
    out = to_pandas_df(df)

    drug_norm = out[drug_col].apply(normalize_therapy)
    gene_norm = out[gene_col].apply(normalize_gene)

    out["normalized_drug_id"] = drug_norm.apply(lambda x: x[0])
    out["normalized_drug_name"] = drug_norm.apply(lambda x: x[1])
    out["normalized_drug_match_type"] = drug_norm.apply(lambda x: x[2])

    out["normalized_gene_id"] = gene_norm.apply(lambda x: x[0])
    out["normalized_gene_name"] = gene_norm.apply(lambda x: x[1])
    out["normalized_gene_match_type"] = gene_norm.apply(lambda x: x[2])

    return out

In [49]:
context_norm_df = add_normalized_columns(
    context_df,
    drug_col="drug_name",
    gene_col="gene_name",
)

results_norm_df = add_normalized_columns(
    results_df,
    drug_col="drug",
    gene_col="gene",
)

In [50]:
results_norm_df

,run_idx,prompt_version,temperature,drug,gene,interaction,normalized_drug_id,normalized_drug_name,normalized_drug_match_type,normalized_gene_id,normalized_gene_name,normalized_gene_match_type
0,0,v1,0,doxorubicin,Bcl-2,False,rxcui:142433,doxorubicin hydrochloride,80.0,hgnc:990,BCL2,60.0
1,0,v1,0,calcipotriol,BCL2,True,rxcui:29365,calcipotriene,80.0,hgnc:990,BCL2,100.0
2,0,v1,0,methylprednisolone sodium succinate,Bcl-2,True,rxcui:203189,methylprednisolone sodium succinate,80.0,hgnc:990,BCL2,60.0
3,0,v1,0,compound 21,Mcl-1,True,drugbank:DB18038,Buloxibutid,60.0,hgnc:32041,ADAMTSL4-AS1,60.0
4,0,v1,0,ZD1839,EGFR,True,rxcui:328134,gefitinib,60.0,hgnc:3236,EGFR,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...
595,0,v1,1,Gefitinib,EGFR,True,rxcui:328134,gefitinib,80.0,hgnc:3236,EGFR,100.0
596,0,v1,1,gefitinib,EGFR,True,rxcui:328134,gefitinib,80.0,hgnc:3236,EGFR,100.0
597,0,v1,1,gefitinib,EGFR,True,rxcui:328134,gefitinib,80.0,hgnc:3236,EGFR,100.0
598,0,v1,1,NaN,NaN,NaN,None,None,NaN,None,None,NaN


### Normalized Stats

Do the stats

In [51]:
# Correct Temps
temps = [0, 0.2, 0.4, 0.6, 0.8, 1]

n_contexts = len(context_norm_df)
n_runs_total = len(results_norm_df) // n_contexts

assert n_runs_total == len(temps)

results_norm_df = results_norm_df.copy()
results_norm_df["temperature"] = [
    temp
    for temp in temps
    for _ in range(n_contexts)
]


temps = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
n_contexts = len(context_norm_df)

assert len(results_df) == n_contexts * len(temps)

results_df = results_df.copy()
results_df["temperature"] = [
    temp
    for temp in temps
    for _ in range(n_contexts)
]

results_norm_df = results_norm_df.copy()
results_norm_df["temperature"] = results_df["temperature"].values

results_norm_df.groupby("temperature")["drug"].nunique()



temperature
0.0    72
0.2    72
0.4    72
0.6    72
0.8    72
1.0    72
Name: drug, dtype: int64

In [52]:
import pandas as pd

ctx = context_norm_df.copy()
pred = results_norm_df.copy()

n_runs = len(pred) // len(ctx)
assert len(pred) == len(ctx) * n_runs

truth = pd.concat([ctx] * n_runs, ignore_index=True)

def clean_id(x):
    if pd.isna(x) or str(x).strip() == "":
        return None
    return str(x).strip().lower()

truth_drug = truth["normalized_drug_id"].map(clean_id)
truth_gene = truth["normalized_gene_id"].map(clean_id)

pred_drug = pred["normalized_drug_id"].map(clean_id)
pred_gene = pred["normalized_gene_id"].map(clean_id)

eval_df = pred.copy()
eval_df["truth_drug_id"] = truth_drug
eval_df["truth_gene_id"] = truth_gene
eval_df["pred_drug_id"] = pred_drug
eval_df["pred_gene_id"] = pred_gene

eval_df["drug_match"] = eval_df["pred_drug_id"].eq(eval_df["truth_drug_id"])
eval_df["gene_match"] = eval_df["pred_gene_id"].eq(eval_df["truth_gene_id"])
eval_df["pair_match"] = eval_df["drug_match"] & eval_df["gene_match"]

eval_df["drug_predicted"] = eval_df["pred_drug_id"].notna()
eval_df["gene_predicted"] = eval_df["pred_gene_id"].notna()
eval_df["pair_predicted"] = (
    eval_df["pred_drug_id"].notna() &
    eval_df["pred_gene_id"].notna()
)

def extraction_metrics(match_col, predicted_col):
    correct = eval_df[match_col]
    predicted = eval_df[predicted_col]

    tp = (predicted & correct).sum()
    fp = (predicted & ~correct).sum()
    fn = (~predicted).sum()

    total = len(eval_df)

    accuracy = correct.mean()
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0
    )

    return {
        "n": total,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

metrics_df = pd.DataFrame([
    {"task": "drug", **extraction_metrics("drug_match", "drug_predicted")},
    {"task": "gene", **extraction_metrics("gene_match", "gene_predicted")},
    {"task": "pairwise", **extraction_metrics("pair_match", "pair_predicted")},
])

metrics_df

,task,n,tp,fp,fn,accuracy,precision,recall,f1
0,drug,600,174,294,132,0.29,0.371795,0.568627,0.449612
1,gene,600,432,96,72,0.72,0.818182,0.857143,0.837209
2,pairwise,600,132,318,150,0.22,0.293333,0.468085,0.360656


### Chart

Make a graph to visualize the scores

In [ ]:
import pandas as pd
import plotly.express as px

plot_df = metrics_df.melt(
    id_vars=["task"],
    value_vars=["precision", "recall", "f1"],
    var_name="metric",
    value_name="score"
)

fig = px.bar(
    plot_df,
    x="metric",
    y="score",
    color="task",
    barmode="group",
    title="LLM Extraction Performance",
    text="score",
    category_orders={
        "metric": ["precision", "recall", "f1"]
    }
)

fig.update_traces(
    texttemplate="%{text:.3f}",
    textposition="outside"
)

fig.update_yaxes(range=[0, 1.05])

fig.update_layout(
    uniformtext_minsize=10,
    uniformtext_mode="hide"
)

fig.show()